# Lab 3 · Deduplicación y taxonomía

**Sesión Norma+ · Diplomado de Ciencia de Datos e IA para Ciencias Sociales**

Distintas resoluciones repiten el mismo criterio con otras palabras. Si no los fusionamos, la base dice «hay 22 criterios» cuando hay 20, y el abogado no ve que un criterio se sostiene en dos resoluciones. Ese dato es justo el que más le interesa.

Deduplicamos en tres capas, de la más barata a la más cara, y dejamos la última palabra a reglas que podemos auditar:

| Capa | Herramienta | Rol | Costo |
|---|---|---|---|
| 1 | Embeddings | **Proponer** pares candidatos | Gratis (modelo abierto en Colab) |
| 2 | Claude como juez | **Decidir** si expresan el mismo criterio | Una llamada por candidato |
| 3 | Reglas en Python | **Vetar** fusiones inconsistentes | Gratis y auditable |

Modelo: `claude-haiku-4-5`. Costo esperado: menos de USD 0.02.

## 0 · Preparación

In [ ]:
!pip install -q anthropic pydantic pandas sentence-transformers

In [ ]:
# Descarga del dataset desde el repositorio público del curso (git viene instalado en Colab).
import os, shutil, zipfile
REPO_URL = "https://github.com/benjov/laboratorio-norma-notebooks"

if not os.path.exists("norma_mini_dataset"):
    !git clone -q --depth 1 "$REPO_URL" repo_curso
    if os.path.exists("repo_curso/norma_mini_dataset"):
        shutil.copytree("repo_curso/norma_mini_dataset", "norma_mini_dataset")
    else:
        print("No se pudo descargar de GitHub. Sube norma_mini_dataset.zip a mano:")
        from google.colab import files
        files.upload()
        zipfile.ZipFile("norma_mini_dataset.zip").extractall(".")
print(sorted(os.listdir("norma_mini_dataset")))

In [ ]:
from google.colab import userdata
import anthropic
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

In [ ]:
# Guardián de presupuesto: envuelve al cliente, suma el costo de cada llamada y se detiene al llegar al límite.
# Precios en USD por millón de tokens (entrada, salida). Un modelo no listado usa un precio conservador.
PRECIOS = {"claude-haiku-4-5": (1.0, 5.0)}
PRECIO_CONSERVADOR = (5.0, 25.0)

class Presupuesto:
    def __init__(self, client, limite_usd):
        self.client, self.limite, self.gastado, self.llamadas = client, limite_usd, 0.0, 0
        self.registro = []  # una fila por llamada: modelo, tokens y costo

    def crear(self, **kwargs):
        if self.gastado >= self.limite:
            raise RuntimeError(f"Límite de USD {self.limite:.2f} alcanzado en este notebook.")
        r = self.client.messages.create(**kwargs)
        p_in, p_out = PRECIOS.get(kwargs["model"], PRECIO_CONSERVADOR)
        costo = (r.usage.input_tokens * p_in + r.usage.output_tokens * p_out) / 1e6
        self.gastado += costo
        self.llamadas += 1
        self.registro.append({"modelo": kwargs["model"], "entrada": r.usage.input_tokens,
                              "salida": r.usage.output_tokens, "costo": costo})
        return r

    def __repr__(self):
        return f"{self.llamadas} llamadas · USD {self.gastado:.4f} de {self.limite:.2f}"

In [ ]:
import re, json, hashlib, itertools
from pathlib import Path
import numpy as np
import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from sentence_transformers import SentenceTransformer

DATA = Path("norma_mini_dataset")
SALIDAS = DATA / "salidas"
SALIDAS.mkdir(exist_ok=True)
MODELO = "claude-haiku-4-5"
api = Presupuesto(client, limite_usd=0.50)

taxonomia = {n["id"]: n["ruta_tema"] for n in json.load(open(DATA / "taxonomia.json", encoding="utf-8"))}

def a_conjunto(x):
    """Artículos como conjunto, venga como lista (Lab 2) o como texto 'LFCE 63; LFCE 64' (gold)."""
    if isinstance(x, list):
        return set(x)
    if isinstance(x, str) and x.strip():
        return {a.strip() for a in x.split(";")}
    return set()

gold = pd.read_csv(DATA / "gold_criterios.csv")
gold["articulos"] = gold.articulos.apply(a_conjunto)
print(len(gold), "criterios en el gold ·", gold.duplicado_de.notna().sum(), "marcados como duplicados")

## 1 · Embeddings: proponer candidatos

Convertimos cada criterio en un vector de 384 dimensiones con un modelo multilingüe abierto. Dos criterios con significado parecido quedan cerca, aunque usen palabras distintas. Medimos la cercanía con la **similitud coseno**.

Embebemos la **abstracción** (`criterio`), no la cita. La cita arrastra detalles del caso; la abstracción captura la regla.

In [ ]:
emb_modelo = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

def similitudes(textos):
    v = emb_modelo.encode(list(textos), normalize_embeddings=True)
    return v @ v.T   # con vectores normalizados, el producto punto es la similitud coseno

S = similitudes(gold.criterio)
ids = gold.id_criterio.tolist()
pares_todos = pd.DataFrame([{"a": ids[i], "b": ids[j], "sim": float(S[i, j])}
                            for i, j in itertools.combinations(range(len(ids)), 2)], columns=["a", "b", "sim"])

dups_gold = {tuple(sorted((r.id_criterio, r.duplicado_de))) for r in gold.itertuples() if pd.notna(r.duplicado_de)}
pares_todos["es_dup"] = [tuple(sorted((a, b))) in dups_gold for a, b in zip(pares_todos.a, pares_todos.b)]
pares_todos.sort_values("sim", ascending=False).head(10)

### ✏️ Ejercicio 1 · ¿Dónde cortar?

Mira la tabla: los duplicados reales (`es_dup = True`) deberían estar arriba, pero no solos. Elige `UMBRAL` para que **ambos duplicados entren como candidatos** con el menor número posible de pares falsos.

El umbral solo propone; un candidato de más cuesta una llamada al juez, y un duplicado que no entra se pierde para siempre. Por eso conviene pecar de generoso.

> En producción, Norma+ usa 0.78 con otro modelo de embeddings. Los umbrales **no se transfieren entre modelos**: cada uno distribuye las similitudes a su manera.

In [ ]:
UMBRAL = round(pares_todos[pares_todos.es_dup].sim.min() - 0.05, 2)  # 0.05 de holgura bajo el duplicado más difícil

In [ ]:
candidatos = pares_todos[pares_todos.sim >= UMBRAL].copy()
print(f"UMBRAL = {UMBRAL} → {len(candidatos)} candidatos de {len(pares_todos)} pares posibles; "
      f"duplicados reales entre ellos: {candidatos.es_dup.sum()} de {len(dups_gold)}")
assert candidatos.es_dup.sum() == len(dups_gold), "Algún duplicado real quedó fuera: baja el umbral"
print("✅ Ambos duplicados son candidatos")
candidatos.sort_values("sim", ascending=False)

## 2 · Claude como juez

Para cada candidato preguntamos si los dos criterios expresan **la misma regla jurídica**. Pedimos confianza y razón, igual que en el validador. Solo aceptamos `mismo_criterio = True` con confianza ≥ 0.85.

In [ ]:
class Juicio(BaseModel):
    mismo_criterio: bool = Field(description="True si ambos expresan la misma regla jurídica, aunque la redacción difiera.")
    confianza: float = Field(description="Entre 0 y 1.")
    razon: str = Field(description="Una oración.")

SISTEMA_JUEZ = """Decide si dos criterios jurídicos expresan LA MISMA REGLA, aunque estén redactados distinto
o provengan de resoluciones diferentes. No son el mismo criterio si uno es más amplio, si tratan aspectos
distintos de un mismo tema, o si uno es condición o consecuencia del otro."""

esq = Juicio.model_json_schema(); esq.pop("$defs", None)
HERR_JUEZ = {"name": "registrar_juicio", "description": "Registra si los criterios son el mismo.", "input_schema": esq}

CACHE_RUTA = SALIDAS / "cache_lab3.json"
cache = json.load(open(CACHE_RUTA, encoding="utf-8")) if CACHE_RUTA.exists() else {}

def juzgar(criterio_a, criterio_b):
    usuario = f"CRITERIO A:\n{criterio_a}\n\nCRITERIO B:\n{criterio_b}"
    clave = hashlib.sha256((MODELO + SISTEMA_JUEZ + usuario).encode()).hexdigest()
    if clave not in cache:
        r = api.crear(model=MODELO, max_tokens=300, system=SISTEMA_JUEZ, tools=[HERR_JUEZ],
                      tool_choice={"type": "tool", "name": "registrar_juicio"},
                      messages=[{"role": "user", "content": usuario}])
        cache[clave] = next(b.input for b in r.content if b.type == "tool_use")
        json.dump(cache, open(CACHE_RUTA, "w", encoding="utf-8"), ensure_ascii=False)
    try:
        return Juicio.model_validate(cache[clave])
    except ValidationError:
        return None

def juzgar_candidatos(cands, base):
    texto = base.set_index("id_criterio").criterio
    juicios = [juzgar(texto[a], texto[b]) for a, b in zip(cands.a, cands.b)]
    cands = cands.copy()
    cands["juez"] = [bool(j and j.mismo_criterio and j.confianza >= 0.85) for j in juicios]
    cands["razon"] = [j.razon if j else "respuesta inválida" for j in juicios]
    return cands

candidatos = juzgar_candidatos(candidatos, gold)
print(api)
candidatos[["a", "b", "sim", "es_dup", "juez", "razon"]]

## 3 · Reglas duras: la última palabra la tiene Python

El juez puede equivocarse con criterios vecinos: «qué elementos considera la multa» y «cómo debe motivarse la multa» suenan parecidos. Antes de fusionar, exigimos dos condiciones que cualquiera puede auditar:

1. **Mismo tema:** ambos criterios están en el mismo nodo de la taxonomía.
2. **Artículos compatibles:** si ambos invocan artículos, comparten al menos uno; si ninguno invoca artículos, la regla no bloquea.

### ✏️ Ejercicio 2
Completa `articulos_compatibles` y `puede_fusionar`.

In [ ]:
def articulos_compatibles(arts_a, arts_b):
    if not arts_a and not arts_b:
        return True
    return bool(arts_a & arts_b)

def puede_fusionar(fila_a, fila_b):
    return fila_a.id_tema == fila_b.id_tema and articulos_compatibles(fila_a.articulos, fila_b.articulos)

In [ ]:
assert articulos_compatibles(set(), set()) and articulos_compatibles({"LFCE 58"}, {"LFCE 58", "LFCE 59"})
assert not articulos_compatibles({"LFCE 58"}, {"LFCE 59"}) and not articulos_compatibles({"LFCE 58"}, set())
print("✅ Reglas correctas")

def aplicar_reglas(cands, base):
    filas = base.set_index("id_criterio")
    cands = cands.copy()
    cands["reglas"] = [puede_fusionar(filas.loc[a], filas.loc[b]) for a, b in zip(cands.a, cands.b)]
    cands["fusionar"] = cands.juez & cands.reglas
    return cands

candidatos = aplicar_reglas(candidatos, gold)
candidatos[["a", "b", "sim", "es_dup", "juez", "reglas", "fusionar"]]

In [ ]:
pred = {tuple(sorted((a, b))) for a, b in zip(candidatos[candidatos.fusionar].a, candidatos[candidatos.fusionar].b)}
vp = len(pred & dups_gold)
print(f"Fusiones propuestas: {len(pred)} · correctas: {vp} · precisión {vp / max(len(pred), 1):.2f} · exhaustividad {vp / len(dups_gold):.2f}")
vetados = candidatos[candidatos.juez & ~candidatos.reglas]
if len(vetados):
    print("\nEl juez dijo «mismo criterio», pero las reglas lo vetaron:")
    print(vetados[["a", "b", "razon"]].to_string(index=False))

In [ ]:
# Prueba de estrés: dos criterios vecinos, sean o no candidatos. ¿Qué dice el juez y qué dicen las reglas?
estres = pd.DataFrame([{"a": "C09", "b": "C21", "sim": float(S[ids.index("C09"), ids.index("C21")])}])
estres = aplicar_reglas(juzgar_candidatos(estres, gold), gold)
g = gold.set_index("id_criterio")
for x in ("C09", "C21"):
    print(f"{x} [{g.loc[x].id_tema}] {g.loc[x].criterio}")
print()
print(estres[["a", "b", "sim", "juez", "reglas", "fusionar", "razon"]].to_string(index=False))

> **Para discutir:** en la prueba de estrés, ¿acertó el juez? Si se equivocó, las reglas lo corrigieron porque los criterios están en nodos distintos (T21 y T22). Si hubo un veto, mira por qué: ¿temas distintos o artículos distintos? Un veto también puede señalar un **error de taxonomía**. Si dos criterios realmente iguales quedaron en nodos distintos, el problema no está en la deduplicación sino en la clasificación. Por eso las reglas, además de bloquear, sirven para detectar inconsistencias.

## 4 · Fusionar sin perder trazabilidad

Agrupamos los criterios fusionados (si A=B y B=C, los tres forman un grupo) y conservamos **todas las fuentes**: documento, página y cita de cada aparición. El criterio canónico es el de la primera aparición; los demás quedan como respaldo.

In [ ]:
def consolidar(base, cands):
    grupo = {i: i for i in base.id_criterio}
    def raiz(i):
        while grupo[i] != i:
            i = grupo[i]
        return i
    for a, b in zip(cands[cands.fusionar].a, cands[cands.fusionar].b):
        grupo[raiz(b)] = raiz(a)
    base = base.assign(grupo=[raiz(i) for i in base.id_criterio])
    filas = []
    for g, sub in base.groupby("grupo", sort=False):
        canon = sub.iloc[0]
        filas.append({"id": g, "criterio": canon.criterio, "id_tema": canon.id_tema,
                      "ruta_tema": taxonomia[canon.id_tema],
                      "articulos": sorted(set().union(*sub.articulos)),
                      "n_fuentes": sub.documento.nunique(),
                      "fuentes": [{"documento": r.documento, "pagina": int(r.pagina), "cita": r.cita_literal}
                                  for r in sub.itertuples()]})
    return pd.DataFrame(filas)

consolidada = consolidar(gold, candidatos)
print(f"{len(gold)} criterios → {len(consolidada)} tras deduplicar")
for _, c in consolidada[consolidada.n_fuentes > 1].iterrows():
    print(f"\n{c.id} · {c.ruta_tema}\n  {c.criterio}")
    for f in c.fuentes:
        print(f"   ↳ {f['documento']} p. {f['pagina']}: «{f['cita'][:80]}…»")

## 5 · Navegar la base por la taxonomía

Con la base consolidada podemos responder preguntas que un buscador de texto no responde: ¿qué temas concentran criterios?, ¿qué nodos están vacíos?, ¿qué criterios se repiten entre autoridades?

In [ ]:
consolidada["rama"] = consolidada.ruta_tema.str.split(" > ").str[0]
print(consolidada.groupby("rama").size().sort_values(ascending=False).to_string(), "\n")
vacios = sorted(set(taxonomia) - set(consolidada.id_tema))
print("Nodos sin criterios:", ", ".join(f"{t} ({taxonomia[t]})" for t in vacios))

## 6 · Aplicar el pipeline a tu base validada

Si corriste el bloque de validación, aplicamos las tres capas a tus criterios aceptados. Aquí no hay etiquetas: lo que ves es lo que el pipeline propone.

In [ ]:
ruta = SALIDAS / "criterios_validados.jsonl"
if ruta.exists():
    mia = pd.read_json(ruta, lines=True)
    mia = mia[mia.aceptado].reset_index(drop=True)
    mia["id_criterio"] = [f"E{i + 1:02d}" for i in range(len(mia))]
    mia["articulos"] = mia.articulos.apply(a_conjunto)
    mia["pagina"] = mia.pagina.fillna(0).astype(int)
    S2 = similitudes(mia.criterio)
    c2 = pd.DataFrame([{"a": mia.id_criterio[i], "b": mia.id_criterio[j], "sim": float(S2[i, j])}
                       for i, j in itertools.combinations(range(len(mia)), 2)], columns=["a", "b", "sim"])
    c2 = aplicar_reglas(juzgar_candidatos(c2[c2.sim >= UMBRAL], mia), mia)
    base_final = consolidar(mia, c2)
    print(f"{api}\n{len(mia)} criterios aceptados → {len(base_final)} tras deduplicar")
else:
    base_final = consolidada
    print("Sin base validada: se guarda la consolidación del gold.")
base_final.to_json(SALIDAS / "base_consolidada.jsonl", orient="records", lines=True, force_ascii=False)
base_final[["id", "ruta_tema", "n_fuentes", "criterio"]]

## Cierre

| Lo que hicimos | Lo que en Norma+ es |
|---|---|
| Embeddings + umbral | Propuesta de candidatos (umbral 0.78 con el modelo de producción) |
| Claude como juez con confianza ≥ 0.85 | Decisión semántica |
| `puede_fusionar`: mismo tema y artículos compatibles | Fusión en Python con reglas duras |
| Grupos con todas sus fuentes | Un criterio, varias resoluciones, trazabilidad completa |

**Para ciencias sociales:** deduplicar es un problema de medición. Contar dos veces el mismo argumento infla la frecuencia de un tema; fusionar dos argumentos distintos borra una diferencia real. El diseño en capas (proponer barato, decidir caro, vetar con reglas explícitas) sirve para cualquier análisis de contenido a escala.

**Siguiente:** en el Lab 4 evaluamos todo el pipeline contra el gold (precisión, exhaustividad y F1) y revisamos cuánto costó.